In [35]:
import pandas as pd
import os

In [36]:
#Changing and checking working directory
os.chdir(r"C:\Users\selen\OneDrive\Desktop\BD_data")
print(os.getcwd())

C:\Users\selen\OneDrive\Desktop\BD_data


In [37]:
#here it is defining the variables to store the file paths
patient_path = 'C4MPatient.csv'
lab_path = 'C4MLab.csv'
diag_path = 'C4MEncounterdiagnosis.csv'
condition_path = 'C4MHealthCondition.csv'

# Load in files
#uses pandas to read the csv files into separate dataframes. 
patient = pd.read_csv(patient_path, sep='|', on_bad_lines='skip', low_memory=False)
lab = pd.read_csv(lab_path, sep='|', on_bad_lines='skip', low_memory=False)
diag = pd.read_csv(diag_path, sep='|', on_bad_lines='skip', low_memory=False)
condition = pd.read_csv(condition_path, sep='|', on_bad_lines='skip', low_memory=False)

#to examine a summary of the lab dataframe 
#print(lab.info())
#to print the first five rows of the lab dataframe
#print(lab.head())

In [38]:
# Selects the relevant columns from the dataframe 
patient = patient[["Patient_ID","Sex","BirthYear"]]
lab = lab[["Patient_ID","Name_calc","TestResult_calc","PerformedDate"]]
diag = diag[["Patient_ID","DiagnosisText_calc", "DiagnosisCode_calc","DateCreated"]]
condition = condition[["Patient_ID","DiagnosisText_calc","DateCreated"]]

# Define relevant markers for bpd 
relevant_markers = ["TOTAL CHOLESTEROL","HBA1C","FASTING GLUCOSE","LDL","HDL","INR","GLUCOSE TOLERANCE"]

# Filter lab dataset
#this is filtering the lab dataframe to inlcude only the rows where the name_calc column matches one of the test names in relevant_markers.
lab_filtered = lab[lab["Name_calc"].isin(relevant_markers)]
#lab["Name_calc"].value_counts()

# Pivot to wide format so there is one row per patient per test date
lab_pivot = lab_filtered.pivot_table(
    index=["Patient_ID","PerformedDate"],
    columns="Name_calc",
    values="TestResult_calc",
    aggfunc="first"  # Take first row if there are duplicate date/time and patient
).reset_index()

lab_pivot

Name_calc,Patient_ID,PerformedDate,FASTING GLUCOSE,GLUCOSE TOLERANCE,HBA1C,HDL,INR,LDL,TOTAL CHOLESTEROL
0,1002000000000009,2013-05-30 00:00:00,NaN,NaN,5.3,NaN,NaN,NaN,NaN
1,1002000000000015,2010-05-06 00:00:00,NaN,NaN,NaN,1.23,NaN,2.41,4.02
2,1002000000000015,2013-01-10 00:00:00,NaN,NaN,5.4,1.6,NaN,2.62,4.97
3,1002000000000015,2015-05-21 00:00:00,NaN,NaN,5.5,NaN,NaN,NaN,NaN
4,1002000000000016,2012-04-19 00:00:00,5.7,NaN,NaN,1.56,NaN,2.21,4.22
...,...,...,...,...,...,...,...,...,...
1065035,13001000000027114,2015-06-10 00:00:00,5.3,NaN,NaN,1.45,NaN,3.55,5.6
1065036,13001000000027128,2014-09-26 00:00:00,4.7,NaN,NaN,1.07,NaN,3.61,5.6
1065037,13001000000027189,2015-05-11 00:00:00,6.3,NaN,6.8,1.29,NaN,3.68,5.7
1065038,13001000000027197,2012-12-11 00:00:00,5.4,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# PREPROCESSING--------------------------------------------------------------
# Clean diagnosis text and code
diag["DiagnosisText_calc"] = diag["DiagnosisText_calc"].fillna("").str.upper()
diag["DiagnosisCode_calc"] = diag["DiagnosisCode_calc"].astype(str).str.strip()

# Convert string "nan" back to actual NaN
# because the string "nan" is not a proper null value in pandas. 
diag.loc[diag["DiagnosisCode_calc"].str.lower()=="nan","DiagnosisCode_calc"] = pd.NA

# Convert diagnosis dates to date-time format
#errors="coerce" ensures that invalid or unparseable dates are converted to NaT or "Not a Time" 
diag["DateCreated"] = pd.to_datetime(diag["DateCreated"], errors="coerce")

# Remove rows with missing codes or dates
#by doing this only complete records are kept for the later steps 
diag = diag.dropna(subset=["DiagnosisCode_calc","DateCreated"])

# Label bipolar disorder diagnoses
bd_keywords = ["BIPOLAR I DISORDER", "BIPOLAR DISORDER", "OTHER BIPOLAR DISORDERS"]
diag["BipolarDisorder"] = diag["DiagnosisText_calc"].apply(lambda x: int(any(k in x for k in bd_keywords)))
bd_flags = diag.groupby("Patient_ID")["BipolarDisorder"].max().reset_index()

# Find the first-ever diagnosis for each patient
first_any_dx = diag.sort_values(["Patient_ID", "DateCreated"]).groupby("Patient_ID").first().reset_index()
first_any_dx = first_any_dx.rename(columns={"DiagnosisText_calc":"first_any_dx_text",
                                            "DiagnosisCode_calc":"first_any_dx_code",
                                            "DateCreated":"first_any_dx_date"})

# Filter diagnoses by 296 mood disorders
#creates diag_296 by filtering diag dataframe to include only rows where DiagnosisCode_calc starts with "296"
diag_296 = diag[diag["DiagnosisCode_calc"].str.startswith("296")]
first_mood_dx = diag_296.sort_values(["Patient_ID","DateCreated"]).groupby("Patient_ID").first().reset_index()
first_mood_dx = first_mood_dx.rename(columns={"DateCreated":"first_mood_dx_date"})

# Define ICD-9 codes for bipolar disorder
bd_codes = ["296.0","296.1","296.4","296.5","296.6","296.7","296.80","296.89"]

In [ ]:
# Select patients with first diagnosis as bipolar disorder
#select patients with bipolar disorder as first ever diagnosis by filtering the first_any_dx dataframe to include only the patients whose first-ever diagnosis matches one of the bipolar disorder ICD-9 codes
bd_first_clean = first_any_dx[first_any_dx["first_any_dx_code"].isin(bd_codes)].copy()

# Confirm that all patients have bipolar disorder as their first diagnosis
all_bd = bd_first_clean["first_any_dx_code"].isin(bd_codes).all()
print(f"All patients have BD as their first-ever diagnosis: {all_bd}")

# Find patients with first mood disorder diagnosis that is not BD
non_bd_first = first_mood_dx[~first_mood_dx["DiagnosisCode_calc"].isin(bd_codes)]

# Extract all bipolar diagnoses to see when they first occur
all_bd = diag_296[diag_296["DiagnosisCode_calc"].isin(bd_codes)]

# Find the first bipolar disorder diagnosis date for each patient
first_bd_dx = all_bd.sort_values(["Patient_ID","DateCreated"]).groupby("Patient_ID").first().reset_index()
first_bd_dx = first_bd_dx.rename(columns={"DateCreated":"first_BD_dx_date",
                                          "DiagnosisCode_calc":"first_BD_dx_code"})

# Merge patients who did not have BD first with their first BD diagnosis
mdd_to_bd = non_bd_first.merge(first_bd_dx, on="Patient_ID", how="inner")

# Keep patients who were diagnosed with BD after being diagnosed with a mood disorder
mdd_to_bd = mdd_to_bd[mdd_to_bd["first_BD_dx_date"] > mdd_to_bd["first_mood_dx_date"]]

# Add each patient's first-ever diagnosis
mdd_to_bd = mdd_to_bd.merge(first_any_dx[["Patient_ID","first_any_dx_date","first_any_dx_code"]], on="Patient_ID", how="left")

# Ensure MDD was the first-ever diagnosis and not another condition
mdd_to_bd_clean = mdd_to_bd[
    (mdd_to_bd["first_mood_dx_date"] == mdd_to_bd["first_any_dx_date"]) &
    (mdd_to_bd["DiagnosisCode_calc"] == mdd_to_bd["first_any_dx_code"])
]

# Convert lab dates to date-time and drop records with missing dates
lab_pivot = lab_pivot.copy()
lab_pivot["PerformedDate"] = pd.to_datetime(lab_pivot["PerformedDate"], errors="coerce")
lab_pivot = lab_pivot.dropna(subset=["PerformedDate"])

# Merge BD-first group with lab data
bd_lab_merged = lab_pivot.merge(
    bd_first_clean[["Patient_ID","first_any_dx_date","first_any_dx_code","first_any_dx_text"]],
    on="Patient_ID",
    how="inner"
)

bd_lab_merged = bd_lab_merged.rename(columns={
    "first_any_dx_code": "DiagnosisCode_calc",
    "first_any_dx_text": "DiagnosisText_calc"
})

# Tag each lab record as Before or After BD diagnosis
bd_lab_merged["Lab_Timing"] = bd_lab_merged.apply(
    lambda row: "Before" if row["PerformedDate"] < row["first_any_dx_date"] else "After",
    axis=1
)

# Summarize lab timing per patient (before, after, or both relative to BD diagnosis)
lab_timing_summary = bd_lab_merged.groupby("Patient_ID")["Lab_Timing"].unique().reset_index().copy()

# Classify each patient based on lab timing
def classify_lab_timing(timings):
    if "Before" in timings and "After" in timings:
        return "Both Before and After"
    elif "Before" in timings:
        return "Only Before"
    elif "After" in timings:
        return "Only After"
    else:
        return "No Lab Data"

lab_timing_summary["Lab_Data_Timing"] = lab_timing_summary["Lab_Timing"].apply(classify_lab_timing)

# Select patient based on when their lab tests occurred relative to their diagnosis
# Patients who had lab tests both before and after
both_patients = lab_timing_summary[lab_timing_summary["Lab_Data_Timing"] == "Both Before and After"]["Patient_ID"]

# Patients who had lab tests only before diagnosis
only_before_patients = lab_timing_summary[lab_timing_summary["Lab_Data_Timing"] == "Only Before"]["Patient_ID"]

# Patients who had lab tests only after diagnosis
only_after_patients = lab_timing_summary[lab_timing_summary["Lab_Data_Timing"] == "Only After"]["Patient_ID"]

# Count total patients for each lab time
total_patients = bd_first_clean["Patient_ID"].nunique()
patients_with_any_labs = lab_timing_summary["Patient_ID"].nunique()
patients_with_only_before = only_before_patients.nunique()
patients_with_only_after = only_after_patients.nunique()
patients_with_both = both_patients.nunique()

print(f"Total BD patients (first-ever diagnosis): {total_patients}")
print(f"Patients with ANY lab results: {patients_with_any_labs}")
print(f"Patients with labs BEFORE diagnosis: {patients_with_only_before}")
print(f"Patients with labs AFTER diagnosis: {patients_with_only_after}")
print(f"Patients with labs BOTH before and after diagnosis: {patients_with_both}")

# Create final datasets with patient and lab data
bd_labs_all = bd_lab_merged.copy()
bd_labs_only_before = bd_lab_merged[bd_lab_merged["Patient_ID"].isin(only_before_patients)].copy()
bd_labs_only_after = bd_lab_merged[bd_lab_merged["Patient_ID"].isin(only_after_patients)].copy()
bd_labs_both = bd_lab_merged[bd_lab_merged["Patient_ID"].isin(both_patients)].copy()

In [ ]:
#bd_labs_all

In [ ]:
#bd_labs_only_before

In [ ]:
#bd_labs_only_after

In [ ]:
#bd_labs_both

In [ ]:
# MULTIPLE DIAGNOSES CHECK ----------------------------------------------------
# Store patients with multiple diagnoses on the same day as their BD diagnosis
multi_dx_records = []

# Loop through each patient in the BD-first group
for idx, row in bd_first_clean.iterrows():
    patient_id = row["Patient_ID"]
    first_dx_date = row["first_any_dx_date"]

    # All diagnoses for this patient on the same day as BD diagnosis 
    same_day_diagnoses = diag[(diag["Patient_ID"] == patient_id) & (diag["DateCreated"] == first_dx_date)]

    # Drop NaN diagnoses
    same_day_diagnoses_clean = same_day_diagnoses.dropna(subset=["DiagnosisCode_calc"])

    # Identify non-BD diagnoses on the same day
    non_bd_diagnoses = same_day_diagnoses_clean[~same_day_diagnoses_clean["DiagnosisCode_calc"].isin(bd_codes)]

    # If at least one non-BD diagnosis exists, save the patient's same-day diagnoses
    if not non_bd_diagnoses.empty:
        same_day_diagnoses_clean = same_day_diagnoses_clean.copy()
        same_day_diagnoses_clean["BD_Patient_ID"] = patient_id  
        multi_dx_records.append(same_day_diagnoses_clean)

# Combine all records into one table
if multi_dx_records:
    multi_dx_df = pd.concat(multi_dx_records, ignore_index=True)
    print(f"Number of patients with other diagnoses on the same day: {multi_dx_df['Patient_ID'].nunique()}")
else:
    multi_dx_df = pd.DataFrame()
    print("No patients with other diagnoses on the same day were found.")

#multi_dx_df

In [ ]:
# PATIENTS WITH LAB RESULTS AFTER DIAGNOSIS ------------------------------------------
# Combine and filter the patients with lab results "only after" and "both before and after"
after_patient_ids = pd.concat([only_after_patients, both_patients])
bd_labs_after = bd_lab_merged[bd_lab_merged["Patient_ID"].isin(after_patient_ids)].copy()

# Keep lab records that occurred only after the diagnosis date
bd_labs_after = bd_labs_after[bd_labs_after["PerformedDate"] >= bd_labs_after["first_any_dx_date"]]

print(f"Number of patients with lab results after diagnosis date: {bd_labs_after['Patient_ID'].nunique()}")
print(f"Shape of the lab dataset after filtering: {bd_labs_after.shape}")

In [ ]:
bd_labs_after.to_csv('bd_labs_after.csv', index=False)
bd_labs_only_after.to_csv('bd_labs_only_after.csv', index=False)